# RAG-агент для академии селлеров Lamoda

Пайплайн: парсер `academy.lamoda.ru` → гибридный ретривер (BM25 + dense embeddings) → ReAct-агент.

Запускать сверху вниз в Google Colab.

## 1. Зависимости

In [ ]:
!pip install -q requests beautifulsoup4 markdownify rank_bm25 sentence-transformers playwright nest_asyncio openai
!playwright install chromium --with-deps

## 2. Парсер

Сайт `academy.lamoda.ru` работает на Bitrix CMS, значит контент рендерится на сервере и сразу присутствует в HTML-ответе. Playwright нужен не для ожидания JS, а чтобы вести себя как браузер и перехватывать HTTP-ответ через `on_response`.

Важная особенность: URL без trailing slash возвращает страницу-заглушку без `<h1>`. С trailing slash получается нормальная статья. Из-за этого простой `rstrip("/")` ломал весь парсинг.

Discovery двухпроходной: с главной страницы видны только разделы, ссылки на конкретные статьи нужно собирать уже внутри каждого раздела.

In [ ]:
import os, re, asyncio, json, markdownify, nest_asyncio
from bs4 import BeautifulSoup
from playwright.async_api import async_playwright

nest_asyncio.apply()

BASE_URL = "https://academy.lamoda.ru"
ARTICLES_DIR = "articles"
DELAY = 0.5

os.makedirs(ARTICLES_DIR, exist_ok=True)


def normalize_url(href: str) -> str:
    # Bitrix без trailing slash возвращает заглушку вместо статьи
    url = href.split("?")[0].split("#")[0]
    if not url.endswith("/"):
        url += "/"
    return url if url.startswith("http") else BASE_URL + url


def is_article_href(href: str) -> bool:
    return bool(re.match(r"^/articles/[^/]+/[^/]", href))


def collect_links(html: str, deep: bool = False) -> set[str]:
    soup = BeautifulSoup(html, "html.parser")
    urls = set()
    for a in soup.find_all("a", href=True):
        href = a["href"]
        if not is_article_href(href):
            continue
        if deep and len(href.strip("/").split("/")) < 4:
            continue
        urls.add(normalize_url(href))
    return urls


def url_to_filepath(url: str) -> str:
    path = url.replace(BASE_URL, "").strip("/")
    if path.startswith("articles/"):
        path = path[len("articles/"):]
    safe = re.sub(r"[^\w/\-]", "_", path)
    return os.path.join(ARTICLES_DIR, safe + ".md")


def html_to_markdown(html: str) -> str:
    soup = BeautifulSoup(html, "html.parser")
    for tag in soup.find_all(["script", "style", "svg"]):
        tag.decompose()
    for tag in soup.find_all(True, class_=re.compile(
        r"breadcrumb|back-link|cookie|btn-top|datepicker|pagination|"
        r"add-to-favorites|current-article-section-items-header", re.I
    )):
        tag.decompose()
    md = markdownify.markdownify(
        str(soup), heading_style="ATX",
        strip=["img", "button", "form", "input", "iframe", "video"],
    )
    md = re.sub(r"\[([^\]]+)\]\(javascript:[^)]*\)", r"\1", md)
    # дата обновления встраивается в строку с заголовком, убираем её
    md = re.sub(r"\s*Обновлено\s+\d{2}\.\d{2}\.\d{4}\s*", " ", md)
    md = re.sub(r"\s*Содержание статьи\s*", " ", md)
    md = re.sub(r"\n{3,}", "\n\n", md)
    return md.strip()


async def load_page(page, url: str) -> str | None:
    # перехватываем HTTP-ответ сервера, а не DOM после JS
    html_body = None

    async def on_response(response):
        nonlocal html_body
        if (response.url.rstrip("/") == url.rstrip("/")
                and response.status == 200
                and "html" in response.headers.get("content-type", "")):
            try:
                html_body = await response.text()
            except Exception:
                pass

    page.on("response", on_response)
    try:
        await page.goto(url, wait_until="load", timeout=30_000)
        await page.wait_for_timeout(300)
    except Exception as e:
        print(f"  [error] {str(e)[:80]}")
    page.remove_listener("response", on_response)
    return html_body


async def get_article(page, url: str) -> dict | None:
    html_body = await load_page(page, url)
    if not html_body:
        return None

    soup = BeautifulSoup(html_body, "html.parser")
    content_el = soup.find(id="content")
    if not content_el:
        return None

    h1 = content_el.find("h1")
    title = h1.get_text(strip=True) if h1 else ""
    if not title:
        # заглушка Bitrix — нет h1, пропускаем
        return None

    md = html_to_markdown(str(content_el))
    if len(md) < 100:
        return None

    is_section = bool(content_el.find(class_="subsections-article-item"))
    return {"title": title, "markdown": md, "url": url, "is_section": is_section}


async def run_parser_async() -> list[dict]:
    results, errors = [], []

    async with async_playwright() as pw:
        browser = await pw.chromium.launch(
            headless=True,
            args=["--no-sandbox", "--disable-setuid-sandbox", "--disable-dev-shm-usage"],
        )
        context = await browser.new_context(
            viewport={"width": 1280, "height": 800},
            user_agent=(
                "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                "AppleWebKit/537.36 (KHTML, like Gecko) "
                "Chrome/125.0.0.0 Safari/537.36"
            ),
        )
        page = await context.new_page()

        print("Pass 1: собираем разделы...")
        html_main = await load_page(page, BASE_URL + "/")
        section_urls = collect_links(html_main or "", deep=False)
        print(f"  разделов: {len(section_urls)}")

        print("Pass 2: собираем статьи из разделов...")
        article_urls = set(section_urls)
        for sec_url in sorted(section_urls):
            html_sec = await load_page(page, sec_url)
            if html_sec:
                article_urls.update(collect_links(html_sec, deep=True))
            await asyncio.sleep(DELAY)
        print(f"  всего URL: {len(article_urls)}")

        print(f"Pass 3: парсим {len(article_urls)} страниц...")
        for i, url in enumerate(sorted(article_urls), 1):
            art = await get_article(page, url)
            if art:
                fp = url_to_filepath(url)
                os.makedirs(os.path.dirname(fp), exist_ok=True)
                with open(fp, "w", encoding="utf-8") as f:
                    f.write(f"---\nsource: {url}\ntitle: {art['title']}\n---\n\n")
                    f.write(art["markdown"])
                results.append({"url": url, "title": art["title"], "filepath": fp})
                tag = " [раздел]" if art["is_section"] else ""
                print(f"  [{i}] {art['title'][:60]}{tag}")
            else:
                errors.append(url)
            await asyncio.sleep(DELAY)

        await browser.close()

    print(f"\nготово: {len(results)} статей, пропущено: {len(errors)}")
    with open(os.path.join(ARTICLES_DIR, "manifest.json"), "w", encoding="utf-8") as f:
        json.dump(results, f, ensure_ascii=False, indent=2)
    return results


PARSED_ARTICLES = asyncio.run(run_parser_async())
print(f"статей в базе: {len(PARSED_ARTICLES)}")

Проверяем первую статью:

In [ ]:
if PARSED_ARTICLES:
    sample = PARSED_ARTICLES[0]
    print(f"url:   {sample['url']}")
    print(f"title: {sample['title']}")
    print(f"file:  {sample['filepath']}\n")
    with open(sample["filepath"], encoding="utf-8") as f:
        print(f.read()[:600])

## 3. Чанкование

Делим каждую статью на чанки по ~400 слов с перекрытием 50 слов. В каждом чанке храним `source` (URL) и `title` — они попадут в ответ агента.

In [ ]:
CHUNK_SIZE = 400    # слов
CHUNK_OVERLAP = 50


def load_article(filepath: str) -> dict | None:
    try:
        with open(filepath, encoding="utf-8") as f:
            raw = f.read()
        source, title = "", ""
        fm = re.match(r"---\n(.*?)\n---\n", raw, re.DOTALL)
        if fm:
            for line in fm.group(1).splitlines():
                if line.startswith("source:"):
                    source = line.split(":", 1)[1].strip()
                elif line.startswith("title:"):
                    title = line.split(":", 1)[1].strip()
            text = raw[fm.end():].strip()
        else:
            text = raw.strip()
        return {"source": source, "title": title, "text": text}
    except Exception as e:
        print(f"  [ERROR] {filepath}: {e}")
        return None


def chunk_text(text: str, size: int, overlap: int) -> list[str]:
    words = text.split()
    chunks, start = [], 0
    while start < len(words):
        end = min(start + size, len(words))
        chunks.append(" ".join(words[start:end]))
        if end == len(words):
            break
        start += size - overlap
    return chunks


def build_chunks(articles: list[dict]) -> list[dict]:
    all_chunks = []
    for art in articles:
        data = load_article(art["filepath"])
        if data is None or not data["text"].strip():
            continue
        for i, text in enumerate(chunk_text(data["text"], CHUNK_SIZE, CHUNK_OVERLAP)):
            all_chunks.append({
                "text": text,
                "source": data["source"] or art["url"],
                "title": data["title"] or art["title"],
                "chunk_id": f"{art['filepath']}#{i}",
            })
    return all_chunks


CHUNKS = build_chunks(PARSED_ARTICLES)
print(f"Всего чанков: {len(CHUNKS)} из {len(PARSED_ARTICLES)} статей")
if CHUNKS:
    print(f"Пример: '{CHUNKS[0]['title']}' | {CHUNKS[0]['source']}")
    print(f"Текст[:200]: {CHUNKS[0]['text'][:200]}...")

## 4. Гибридный ретривер

BM25 хорошо ловит точные термины (FBS, SKU, API, DataMatrix), dense — перефразированные запросы и синонимы. RRF объединяет оба ранжирования без подбора весов.

In [ ]:
import numpy as np
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer

EMBED_MODEL = "intfloat/multilingual-e5-small"
RRF_K = 60


def tokenize(text: str) -> list[str]:
    return re.findall(r"[а-яёa-z0-9]+", text.lower())


class HybridRetriever:
    def __init__(self, chunks: list[dict]):
        self.chunks = chunks
        print("строим BM25...")
        self.bm25 = BM25Okapi([tokenize(c["text"]) for c in chunks])

        print(f"загружаем {EMBED_MODEL}...")
        self.model = SentenceTransformer(EMBED_MODEL)
        print("считаем эмбеддинги...")
        self.embeddings = self.model.encode(
            [f"passage: {c['text']}" for c in chunks],
            batch_size=32, show_progress_bar=True, normalize_embeddings=True,
        )
        print(f"индекс готов: {len(chunks)} чанков, {self.embeddings.shape[1]}d")

    def search(self, query: str, top_k: int = 5) -> list[dict]:
        # BM25 ранги
        bm25_scores = self.bm25.get_scores(tokenize(query))
        bm25_order = np.argsort(bm25_scores)[::-1]
        bm25_ranks = np.empty_like(bm25_order)
        bm25_ranks[bm25_order] = np.arange(len(bm25_order))

        # dense ранги
        q_emb = self.model.encode([f"query: {query}"], normalize_embeddings=True)
        dense_scores = self.embeddings @ q_emb[0]
        dense_order = np.argsort(dense_scores)[::-1]
        dense_ranks = np.empty_like(dense_order)
        dense_ranks[dense_order] = np.arange(len(dense_order))

        # RRF fusion
        rrf = 1.0 / (RRF_K + bm25_ranks) + 1.0 / (RRF_K + dense_ranks)
        top_idx = np.argsort(rrf)[::-1][:top_k]

        results = []
        for idx in top_idx:
            chunk = self.chunks[idx].copy()
            chunk["score"] = float(rrf[idx])
            results.append(chunk)
        return results


retriever = HybridRetriever(CHUNKS)

In [ ]:
results = retriever.search("регистрация продавца на Lamoda", top_k=3)
for i, r in enumerate(results, 1):
    print(f"[{i}] {r['title']}")
    print(f"     {r['source']}")
    print(f"     {r['text'][:120]}\n")

## 5. ReAct-агент

ReAct-цикл написан руками: `THOUGHT → ACTION → OBSERVATION → ... → FINAL ANSWER`. Ответ LLM парсится regex, при ACTION синхронно вызывается ретривер, результат вставляется как OBSERVATION.

LLM: `openrouter/owl-alpha` это routing-модель OpenRouter, сама выбирает свободного провайдера среди бесплатных. Запасные модели используются если owl-alpha недоступна.

In [ ]:
import time
from openai import OpenAI

OPENROUTER_API_KEY = ""

FREE_MODELS = [
    "openrouter/owl-alpha",
    "google/gemma-4-26b-a4b-it:free",
    "meta-llama/llama-3.1-8b-instruct:free",
    "mistralai/mistral-7b-instruct:free",
    "qwen/qwen3-8b:free",
]
_model_idx = 0

client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=OPENROUTER_API_KEY)

MAX_ITER = 10
TOP_K = 5

SYSTEM_PROMPT = """Ты — консультант академии селлеров Lamoda. Отвечай на вопросы продавцов,
используя базу знаний академии.

У тебя есть один инструмент:
  search_knowledge_base(query: str) — поиск по базе знаний академии Lamoda.
  Возвращает релевантные фрагменты статей со ссылками на источники.

Формат ответа:

THOUGHT: <рассуждение>
ACTION: search_knowledge_base("<запрос>")

После получения OBSERVATION:

THOUGHT: <анализ>
ACTION: search_knowledge_base("<уточняющий запрос>")  ← если нужно

Когда ответ готов:

THOUGHT: <финальное рассуждение>
FINAL ANSWER: <ответ на русском со ссылками [Название](URL)>

Правила:
- минимум один вызов search_knowledge_base перед ответом
- в FINAL ANSWER всегда давай ссылки на источники
- не придумывай факты, только из результатов поиска"""


def format_results(results: list[dict]) -> str:
    if not results:
        return "Ничего не найдено."
    lines = []
    for i, r in enumerate(results, 1):
        lines.append(f"[{i}] {r['title']} — {r['source']}")
        lines.append(r["text"][:600])
        lines.append("")
    return "\n".join(lines)


def parse_action(text: str) -> str | None:
    m = re.search(r'ACTION:\s*search_knowledge_base\(["\'](.+?)["\']\)', text, re.DOTALL)
    return m.group(1).strip() if m else None


def parse_final(text: str) -> str | None:
    m = re.search(r"FINAL ANSWER:\s*(.+)", text, re.DOTALL)
    return m.group(1).strip() if m else None


def llm_call(messages: list) -> str:
    global _model_idx
    for offset in range(len(FREE_MODELS)):
        idx = (_model_idx + offset) % len(FREE_MODELS)
        model = FREE_MODELS[idx]
        for attempt in range(3):
            try:
                resp = client.chat.completions.create(model=model, messages=messages)
                if offset > 0:
                    print(f"  [fallback] {model.split('/')[-1]}")
                    _model_idx = idx
                return resp.choices[0].message.content
            except Exception as e:
                err = str(e)
                if "429" in err and attempt < 2:
                    wait = min(20 * (2 ** attempt), 60)
                    print(f"  [429] {model.split('/')[-1]} ждём {wait}с")
                    time.sleep(wait)
                else:
                    break
    raise RuntimeError("все модели недоступны")


def run_agent(question: str, verbose: bool = True) -> str:
    history = [{"role": "system", "content": SYSTEM_PROMPT}]
    msg = question
    calls = 0

    for i in range(MAX_ITER):
        if verbose:
            print(f"\n[iter {i+1}]")
        history.append({"role": "user", "content": msg})
        text = llm_call(history)
        if verbose:
            print(text)
        history.append({"role": "assistant", "content": text})

        answer = parse_final(text)
        if answer:
            if verbose:
                print(f"\n(search_knowledge_base вызвана {calls} раз)")
            return answer

        query = parse_action(text)
        if query:
            calls += 1
            if verbose:
                print(f"\n>> search_knowledge_base({query!r})")
            obs = format_results(retriever.search(query, top_k=TOP_K))
            if verbose:
                print(obs[:400])
            msg = f"OBSERVATION:\n{obs}"
        else:
            msg = "Продолжай. Если ответ готов — FINAL ANSWER: ... Если нужна информация — ACTION: search_knowledge_base(...)."

    return "[превышен лимит итераций]"


print("агент готов")

## 6. Тест

In [ ]:
q1 = "Как зарегистрироваться на Lamoda как продавец?"
print(f"Вопрос: {q1}\n")
a1 = run_agent(q1)
print(f"\nОтвет:\n{a1}")

In [ ]:
q2 = "Какие есть схемы работы с Lamoda: FBS и FBO, в чём разница?"
print(f"Вопрос: {q2}\n")
a2 = run_agent(q2)
print(f"\nОтвет:\n{a2}")

In [ ]:
q3 = "Что нужно для начала работы с Lamoda?"
print(f"Вопрос: {q3}\n")
a3 = run_agent(q3)
print(f"\nОтвет:\n{a3}")

In [ ]:
q4 = "Как запустить таргетированные акции?"
print(f"Вопрос: {q4}\n")
a4 = run_agent(q4)
print(f"\nОтвет:\n{a4}")

In [ ]:
q5 = "Какие требования для фотографий часов?"
print(f"Вопрос: {q5}\n")
a5 = run_agent(q5)
print(f"\nОтвет:\n{a5}")

In [ ]:
q6 = "Какие особенности отмены заказов через API?"
print(f"Вопрос: {q6}\n")
a6 = run_agent(q6)
print(f"\nОтвет:\n{a6}")

In [ ]:
q7 = "Что должно быть на этикетке обуви?"
print(f"Вопрос: {q7}\n")
a7 = run_agent(q7)
print(f"\nОтвет:\n{a7}")

## 7. Обоснование решений

### Парсинг

Изначально думал, что сайт на Vue SPA и нужен Playwright для ожидания JS. Оказалось на Bitrix CMS, контент server-rendered и уже есть в первом HTML-ответе. Playwright в итоге нужен только чтобы сайт не блокировал по User-Agent/заголовкам, и для `on_response`: перехватываю HTTP-ответ напрямую, это надёжнее чем `page.content()` (тот может вернуть DOM уже после JS-манипуляций).

Главный баг при разработке: URL без trailing slash Bitrix отдаёт страницу-каталог без `<h1>`. С trailing slash получается нормальная статья.

`networkidle` для `wait_until` не подходит — аналитика на странице делает постоянные запросы и idle никогда не наступает. Использую `load` + небольшой `wait_for_timeout`.

### Ретривер

Взял связку BM25 + dense embeddings потому что они дополняют друг друга: BM25 хорошо работает по точным терминам (FBS, артикул, API-токен), dense - по смысловой близости. Для слияния выбрал RRF вместо взвешенного суммирования: не нужно подбирать веса, работает из коробки, и на практике результаты хорошие.

Модель `intfloat/multilingual-e5-small`: мультиязычная (~120MB), обучена специально для retrieval, требует префиксы `passage:` / `query:`. Индекс это numpy-матрица в памяти, cosine similarity через матричное умножение.

### Агент

ReAct написан руками без фреймворков, так понятнее что происходит и проще дебажить. Формат THOUGHT/ACTION/OBSERVATION/FINAL ANSWER парсится двумя regex. Лимит итераций защищает от зацикливания.

Для LLM выбрал OpenRouter: бесплатный, OpenAI-совместимый API (не нужно отдельный SDK). `openrouter/owl-alpha` это routing-модель, сама выбирает доступного провайдера, поэтому нет проблем с 429 конкретной модели. Запасной список на случай если owl-alpha тоже недоступна.

### Что можно улучшить

- cross-encoder re-ranking поверх top-20 (mmarco-mMiniLMv2)
- стемминг через pymorphy2 для русского BM25
- кэш эмбеддингов на диск, а не пересчитывать при каждом перезапуске Colab
- multi-turn: сейчас каждый вопрос независим, история не сохраняется между вызовами `run_agent`